# 03_baseline_xgb_optuna
XGBoost + Optunaによるハイパーパラメータ最適化

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    "/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026"
)
sys.path.append(str(PROJECT_ROOT))

from common.xgboost.xgb_model import run_xgb
from common.xgboost.xgb_model_optuna import run_xgb_optuna
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

SEED = 42
seed_everything(seed=SEED)
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

In [2]:
SCRIPT_NAME = "03_baseline_xgb_optuna"
TODAY = datetime.datetime.now().strftime("%Y%m%d")
LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}.csv"

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

[2026-08-05 11:32:51] [INFO] === [03_baseline_xgb_optuna] 実験開始 ===


In [3]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")
logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")

[2026-08-05 11:32:52] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


In [4]:
def transform_monthly_to_wide_all(monthly_df: pd.DataFrame) -> pd.DataFrame:
    val_cols = [col for col in monthly_df.columns if col not in ["社員ID", "経過月数"]]
    monthly_wide = monthly_df.pivot(index="社員ID", columns="経過月数", values=val_cols)
    monthly_wide.columns = [f"{col}_m{month}" for col, month in monthly_wide.columns]
    return monthly_wide.reset_index()

train_monthly_wide = transform_monthly_to_wide_all(train_monthly)
test_monthly_wide = transform_monthly_to_wide_all(test_monthly)
train_df = pd.merge(train_persona, train_monthly_wide, on="社員ID", how="left")
test_df = pd.merge(test_persona, test_monthly_wide, on="社員ID", how="left")
logger.info(f"結合後 Train Shape: {train_df.shape}, Test Shape: {test_df.shape}")

[2026-08-05 11:32:52] [INFO] 結合後 Train Shape: (2761, 668), Test Shape: (2502, 667)


In [5]:
def build_features(train: pd.DataFrame, test: pd.DataFrame, target_col: str, id_col: str):
    train_proc, test_proc = train.copy(), test.copy()
    non_num_cols = train_proc.select_dtypes(include=["object"]).columns.tolist()
    if id_col in non_num_cols:
        non_num_cols.remove(id_col)
    train_proc = train_proc.drop(columns=non_num_cols, errors="ignore")
    test_proc = test_proc.drop(columns=non_num_cols, errors="ignore")
    X_train = train_proc.drop(columns=[target_col, id_col], errors="ignore")
    y_train = train_proc[target_col]
    X_test = test_proc.drop(columns=[id_col], errors="ignore")
    return {"X_train": X_train, "y_train": y_train, "X_test": X_test}, test_proc[id_col]

input_data, test_ids = build_features(train_df, test_df, target_col=TARGET_COL, id_col=ID_COL)
logger.info(f"X_train Shape: {input_data['X_train'].shape}")

[2026-08-05 11:32:52] [INFO] X_train Shape: (2761, 3)


In [6]:
xgb_opt_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR),
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "early_stopping_rounds": 50,
    "n_trials": 20,
    "verbose": False,
}
logger.info("Optunaパラメータ:")
for key, value in xgb_opt_params.items():
    logger.info(f"  {key}: {value}")

[2026-08-05 11:32:52] [INFO] Optunaパラメータ:
[2026-08-05 11:32:52] [INFO]   n_splits: 5
[2026-08-05 11:32:52] [INFO]   seed: 42
[2026-08-05 11:32:52] [INFO]   save_dir: /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/saved_models/20260805/03_baseline_xgb_optuna
[2026-08-05 11:32:52] [INFO]   objective: binary:logistic
[2026-08-05 11:32:52] [INFO]   eval_metric: logloss
[2026-08-05 11:32:52] [INFO]   early_stopping_rounds: 50
[2026-08-05 11:32:52] [INFO]   n_trials: 20
[2026-08-05 11:32:52] [INFO]   verbose: False


In [7]:
logger.info("--- Optuna XGBoost チューニング開始 ---")
opt_xgb_res, best_xgb_params = run_xgb_optuna(data=input_data, params=xgb_opt_params)
logger.info(f"XGBoost Optuna Best Score: {opt_xgb_res['best_score']:.4f}")
logger.info(f"Best Parameters: {best_xgb_params}")

[2026-08-05 11:32:52] [INFO] --- Optuna XGBoost チューニング開始 ---


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "save_dir", "verbose" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "save_dir", "verbose" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "save_dir", "verbose" } are

[2026-08-05 11:32:55] [INFO] XGBoost Optuna Best Score: 0.6696
[2026-08-05 11:32:55] [INFO] Best Parameters: {'n_splits': 5, 'seed': 42, 'save_dir': '/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/saved_models/20260805/03_baseline_xgb_optuna', 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'early_stopping_rounds': 50, 'n_trials': 20, 'verbose': False, 'learning_rate': 0.19957214368558046, 'max_depth': 8, 'min_child_weight': 20, 'subsample': 0.613993876371891, 'colsample_bytree': 0.7743793303666697, 'alpha': 0.038807110612329394, 'lambda': 3.4567385004228085e-06}


In [8]:
logger.info("--- XGBoost 本学習実行 ---")
xgb_res, _ = run_xgb(data=input_data, params=best_xgb_params)
xgb_cv = calculate_logloss(input_data["y_train"], xgb_res["oof_preds"])
logger.info(f"XGBoost CV Score (LogLoss): {xgb_cv:.4f}")

[2026-08-05 11:32:55] [INFO] --- XGBoost 本学習実行 ---
[2026-08-05 11:32:55] [INFO] XGBoost CV Score (LogLoss): 0.6696


/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "n_trials", "verbose" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "n_trials", "verbose" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/.venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [11:32:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "enable_categorical", "n_trials", "verbose" } are

In [9]:
sub = pd.DataFrame({ID_COL: test_ids, TARGET_COL: xgb_res["test_preds"]})
sub.to_csv(SUBMISSION_PATH, index=False)
logger.info(f"提出ファイルを保存しました: {SUBMISSION_PATH}")
logger.info("=== 実験完了 ===")
print(f"\n提出ファイル: {SUBMISSION_PATH}")
print(f"Optuna Best Score: {opt_xgb_res['best_score']:.4f}")
print(f"Final CV Score: {xgb_cv:.4f}")

[2026-08-05 11:32:55] [INFO] 提出ファイルを保存しました: /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/20260805_03_baseline_xgb_optuna.csv
[2026-08-05 11:32:55] [INFO] === 実験完了 ===

提出ファイル: /Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026/data/output/20260805/20260805_03_baseline_xgb_optuna.csv
Optuna Best Score: 0.6696
Final CV Score: 0.6696
